# Calculate ETCCDI climate indices — E3SMv3

This notebook calculates annual ETCCDI climate indices for E3SMv3 experiments.
Two different S3 data sources are used:

| Scenario | Bucket | Style | Members |
|---|---|---|---|
| G6-1.5K-HiLLA | `reflective-persistent-prod-large` | Native ne30pg2 grid, requires local download + ncremap | 0101, 0151, 0201 |
| SSP245 | `reflective-persistent-prod-large` | Native ne30pg2 grid, requires local download + ncremap | 0101, 0151, 0201 |
| G6-1.5K-SAI | `reflective-persistent-prod/alistairduffey` | Pre-regridded 180×360 lat/lon | 001, 002, 003 |

The HiLLA and SSP245 files cannot be opened directly from S3 (old NetCDF format on native unstructured grid).
Each file is downloaded to a local temp directory, regridded to a regular 180×360 lat/lon grid with
`ncremap` using the provided ESMF weight map, then processed with xclim. Raw files are deleted
immediately after regridding to keep disk usage low.

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html

In [2]:
import xarray as xr
import numpy as np
import gc
import os
import subprocess
from os.path import join
import xclim.indices
import s3fs
import fsspec
import dask

In [3]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

model = 'E3SMv3'

# --- G6-1.5K-HiLLA and SSP245: reflective-persistent-prod-large ---
# Files are on the native ne30pg2 unstructured grid; must be downloaded
# locally and regridded with ncremap before use.
E3SM_S3_BASE = 's3://reflective-persistent-prod-large/E3SMv3/G6-1.5K-HiLLA/'
MAP_S3_PATH  = f'{E3SM_S3_BASE}maps/map_ne30pg2_to_cmip6_180x360_aave.20200201.nc'
MAP_LOCAL    = '/tmp/e3sm_regrid/map_ne30pg2_to_180x360.nc'

# member_dir: subdirectory under E3SM_S3_BASE for each scenario/member
E3SM_SCENARIOS = {
    'G6-1.5K-HiLLA': {
        'r1': 'v3.LR.ssp245.g6_hilla.sai.0101',
        'r2': 'v3.LR.ssp245.g6_hilla.sai.0151',
        'r3': 'v3.LR.ssp245.g6_hilla.sai.0201',
    },
    'SSP245': {
        'r1': 'v3.LR.ssp245_0101',
        'r2': 'v3.LR.ssp245_0151',
        'r3': 'v3.LR.ssp245_0201',
    },
}

# --- G6-1.5K-SAI: alistairduffey bucket ---
# Files are already on a regular 180x360 lat/lon grid.
ALISTAIR_S3_BASE = 's3://reflective-persistent-prod/alistairduffey/E3SMv3/G6-1.5K-SAI/'
ALISTAIR_MEMBERS = {'r1': '001', 'r2': '002', 'r3': '003'}

OUTPUT_ROOT = f'./out_data/ETCCDI_indices_annual/{model}/'
TEMP_DIR    = '/tmp/e3sm_regrid'

fs = s3fs.S3FileSystem(anon=False)

# 180x360 grid: 365 x 180 x 360 x 4 B ≈ 94 MB per chunk
CHUNKS = {'time': 365}

In [4]:
# -------------------------------------------------------
# Setup: directories, ncremap check, map file download
# -------------------------------------------------------

os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Verify ncremap (part of NCO) is available
_result = subprocess.run(['which', 'ncremap'], capture_output=True, text=True)
if _result.returncode != 0:
    raise RuntimeError(
        'ncremap not found. Install NCO with: conda install -c conda-forge nco'
    )
print(f'ncremap: {_result.stdout.strip()}')

# Download the ESMF regridding weight map once
if not os.path.exists(MAP_LOCAL):
    print('Downloading regridding map...')
    fs.get(MAP_S3_PATH.replace('s3://', ''), MAP_LOCAL)
    print(f'Map saved to {MAP_LOCAL}')
else:
    print(f'Map already present: {MAP_LOCAL}')

ncremap: /srv/conda/envs/notebook/bin/ncremap
Map already present: /tmp/e3sm_regrid/map_ne30pg2_to_180x360.nc


In [5]:
# -------------------------------------------------------
# Helper functions
# -------------------------------------------------------

SECONDS_PER_DAY = 86400
M_TO_MM = 1000

def to_mm_day(da, varname=None):
    """
    Convert precipitation rate to mm/day.
    Handles kg m-2 s-1 and m s-1 units.
    E3SM PRECT is in m/s (total precip rate).
    """
    units = (da.attrs.get('units') or '').strip().lower().replace(' ', '')
    is_kg_m2_s = units in {'kgm-2s-1', 'kg/m^2/s', 'kg/m2/s', 'kgm^-2s^-1', 'kgm**-2s**-1'}
    is_m_s     = units in {'ms-1', 'm/s', 'm s-1', 'm/s-1'} or units == 'ms^-1'
    if is_kg_m2_s:
        out = da * SECONDS_PER_DAY
    elif is_m_s:
        out = da * SECONDS_PER_DAY * M_TO_MM
    else:
        if varname is not None and varname.upper() in {'PRECT', 'PRECC', 'PRECL', 'PR'}:
            # E3SM PRECT has units m/s — convert to mm/day
            out = da * SECONDS_PER_DAY * M_TO_MM
        else:
            return da
    out = out.assign_attrs(da.attrs)
    out.attrs['units'] = 'mm/day'
    return out


def download_regrid_files(s3_paths, temp_dir, map_local):
    """
    For each S3 path:
      1. Download the raw file to temp_dir.
      2. Regrid with ncremap using map_local.
      3. Delete the raw file immediately to save disk space.
    Returns a list of local paths to the regridded files.
    Skips files already regridded (safe to re-run).
    """
    local_paths = []
    for s3_path in s3_paths:
        fname        = os.path.basename(s3_path)
        local_raw    = join(temp_dir, f'raw_{fname}')
        local_regrid = join(temp_dir, f'rg_{fname}')
        if not os.path.exists(local_regrid):
            print(f'    Downloading {fname}...')
            fs.get(s3_path, local_raw)
            print(f'    Regridding  {fname}...')
            result = subprocess.run(
                ['ncremap', '-m', map_local, '-i', local_raw, '-o', local_regrid],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(
                    f'ncremap failed for {fname}:\n{result.stderr}'
                )
            os.remove(local_raw)
        local_paths.append(local_regrid)
    return local_paths


def download_files(s3_paths, temp_dir):
    """
    Download files from S3 to temp_dir without regridding.
    Used for alistairduffey SAI files (CDF-5 format, already on 180x360 grid).
    Skips files already downloaded (safe to re-run).
    Returns list of local file paths.
    """
    local_paths = []
    for s3_path in s3_paths:
        fname = os.path.basename(s3_path)
        local = join(temp_dir, f'sai_{fname}')
        if not os.path.exists(local):
            print(f'    Downloading {fname}...')
            fs.get(s3_path, local)
        local_paths.append(local)
    return local_paths

## Temperature indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| TREFHTMX (Tx) | SU | Summer days | Number of days when Tx $>$ 25$^\circ$C | days per year | Fixed threshold |
| TREFHTMX (Tx) | ID | Ice days | Number of days when Tx $<$ 0$^\circ$C | days per year | Fixed threshold |
| TREFHTMX (Tx) | TXn | Coldest days | Annual minimum daily maximum temperature | $^\circ$C | Fixed Index |
| TREFHTMX (Tx) | TXx | Warmest days | Annual maximum daily maximum temperature | $^\circ$C | Fixed Index |
| TREFHTMN (Tn) | TNn | Coldest night | Annual minimum daily minimum temperature | $^\circ$C | Fixed Index |
| TREFHTMN (Tn) | TNx | Warmest night | Annual maximum daily minimum temperature | days per year | Fixed Index |
| TREFHTMN (Tn) | FD | Frost days | Number of days when Tn < 0$^\circ$C | days per year | Fixed threshold |
| TREFHTMN (Tn) | TN | Tropical nights | Number of days when Tn $>$ 20$^\circ$C | days per year | Fixed threshold |

### G6-1.5K-HiLLA and SSP245 (reflective-persistent-prod-large, native grid → ncremap)

Temperature variables: `TREFHTMX` (tasmax), `TREFHTMN` (tasmin) — units: K  
Path: `{E3SM_S3_BASE}{member_dir}/day/{var}/gn/*/{var}_*.nc`

In [10]:
for scenario, members in E3SM_SCENARIOS.items():
    for member, member_dir in members.items():
        print(f'Processing scenario={scenario}, member={member} ({member_dir})')

        tmax_s3 = sorted(fs.glob(
            f'{E3SM_S3_BASE}{member_dir}/day/TREFHTMX/gn/*/TREFHTMX_*.nc'
        ))
        tmin_s3 = sorted(fs.glob(
            f'{E3SM_S3_BASE}{member_dir}/day/TREFHTMN/gn/*/TREFHTMN_*.nc'
        ))
        print(f'  Found {len(tmax_s3)} TREFHTMX files, {len(tmin_s3)} TREFHTMN files')

        print('  Downloading and regridding TREFHTMX...')
        tmax_local = download_regrid_files(tmax_s3, TEMP_DIR, MAP_LOCAL)
        print('  Downloading and regridding TREFHTMN...')
        tmin_local = download_regrid_files(tmin_s3, TEMP_DIR, MAP_LOCAL)

        tasmax_ds = xr.open_mfdataset(
            tmax_local, combine='nested', concat_dim='time',
            chunks=CHUNKS, coords='minimal', compat='override'
        )
        tasmin_ds = xr.open_mfdataset(
            tmin_local, combine='nested', concat_dim='time',
            chunks=CHUNKS, coords='minimal', compat='override'
        )

        # Deduplicate before slicing (SSP245 files span 2015–2084 with possible
        # overlapping timestamps at decade boundaries)
        tasmax_ds = tasmax_ds.drop_duplicates('time')
        tasmin_ds = tasmin_ds.drop_duplicates('time')

        tasmax_ds = tasmax_ds.sel(time=slice('2035', '2084'))
        tasmin_ds = tasmin_ds.sel(time=slice('2035', '2084'))

        # Ensure units are set (E3SM output is in K)
        tx = tasmax_ds['TREFHTMX'].assign_attrs({'units': 'K'})
        tn = tasmin_ds['TREFHTMN'].assign_attrs({'units': 'K'})

        out_dir = join(OUTPUT_ROOT, scenario, member)
        os.makedirs(out_dir, exist_ok=True)

        # --- Pass 1: tasmax-based indices ---
        su, id_, txx, txn = dask.compute(
            xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS'),
            xclim.indices.ice_days(tx, thresh='0 degC', freq='YS'),
            xclim.indices.tx_max(tx, freq='YS'),
            xclim.indices.tx_min(tx, freq='YS'),
        )
        su.to_netcdf(join(out_dir, 'SU.nc'));   del su
        id_.to_netcdf(join(out_dir, 'ID.nc'));  del id_
        txx.to_netcdf(join(out_dir, 'TXX.nc')); del txx
        txn.to_netcdf(join(out_dir, 'TXN.nc')); del txn
        gc.collect()

        # --- Pass 2: tasmin-based indices ---
        fd, tn_, tnx, tnn = dask.compute(
            xclim.indices.frost_days(tn, thresh='0 degC', freq='YS'),
            xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS'),
            xclim.indices.tn_max(tn, freq='YS'),
            xclim.indices.tn_min(tn, freq='YS'),
        )
        fd.to_netcdf(join(out_dir, 'FD.nc'));   del fd
        tn_.to_netcdf(join(out_dir, 'TN.nc'));  del tn_
        tnx.to_netcdf(join(out_dir, 'TNX.nc')); del tnx
        tnn.to_netcdf(join(out_dir, 'TNN.nc')); del tnn

        del tasmax_ds, tasmin_ds, tx, tn
        gc.collect()

        # Clean up regridded temp files to free disk space
        for p in tmax_local + tmin_local:
            if os.path.exists(p):
                os.remove(p)

        print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=G6-1.5K-HiLLA, member=r1 (v3.LR.ssp245.g6_hilla.sai.0101)
  Found 10 TREFHTMX files, 10 TREFHTMN files
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-HiLLA/r1
Processing scenario=G6-1.5K-HiLLA, member=r2 (v3.LR.ssp245.g6_hilla.sai.0151)
  Found 10 TREFHTMX files, 10 TREFHTMN files
    Regridding  TREFHTMX_203501_203912.nc...
    Regridding  TREFHTMX_204001_204412.nc...
    Regridding  TREFHTMX_204501_204912.nc...
    Regridding  TREFHTMX_205001_205412.nc...
    Regridding  TREFHTMX_205501_205912.nc...
    Regridding  TREFHTMX_206001_206412.nc...
    Regridding  TREFHTMX_206501_206912.nc...
    Regridding  TREFHTMX_207001_207412.nc...
    Regridding  TREFHTMX_207501_207912.nc...
    Regridding  TREFHTMX_208001_208412.nc...
    Regridding  TREFHTMN_203501_203912.nc...
    Regridding  TREFHTMN_204001_204412.nc...
    Regridding  TREFHTMN_204501_204912.nc...
    Regridding  TREFHTMN_205001_205412.nc...
    Regridding  TREFHTMN_205501_2

### G6-1.5K-SAI (alistairduffey bucket, pre-regridded 180×360)

Temperature variables: `TREFHTMX` in `daily_T_max/{member}/`, `TREFHTMN` in `daily_T_min/{member}/`  
Run the diagnostic cell first to confirm dimension names and variable structure.

In [16]:
# Diagnostic: download first SAI file locally, then inspect structure.
# CDF-5 format (PnetCDF) cannot be streamed via fsspec — must use a local path
# with engine='netcdf4' (the C netCDF library handles CDF-1/2/5 and HDF5).

_files = sorted(fs.glob(f'{ALISTAIR_S3_BASE}daily_T_max/001/*.nc'))
print(f'Found {len(_files)} files in daily_T_max/001/')
if _files:
    _fname = os.path.basename(_files[0])
    _local = os.path.join(TEMP_DIR, f'diag_{_fname}')
    if not os.path.exists(_local):
        print(f'Downloading {_fname}...')
        fs.get(_files[0], _local)
    _ds = xr.open_dataset(_local, engine='netcdf4')
    print('dims:     ', dict(_ds.dims))
    print('coords:   ', list(_ds.coords))
    print('data_vars:', list(_ds.data_vars))
    print()
    print(_ds)
    _ds.close()
    os.remove(_local)

Found 7 files in daily_T_max/001/
dims:      {'lat': 180, 'lon': 360, 'nbnd': 2, 'time': 1825}
coords:    ['lat', 'lon', 'time']
data_vars: ['lat_bnds', 'lon_bnds', 'gw', 'area', 'TREFHTMX', 'time_bnds']

<xarray.Dataset> Size: 474MB
Dimensions:    (lat: 180, lon: 360, nbnd: 2, time: 1825)
Coordinates:
  * lat        (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon        (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * time       (time) object 15kB 2035-01-02 00:00:00 ... 2040-01-01 00:00:00
Dimensions without coordinates: nbnd
Data variables:
    lat_bnds   (lat, nbnd) float64 3kB ...
    lon_bnds   (lon, nbnd) float64 6kB ...
    gw         (lat) float64 1kB ...
    area       (lat, lon) float64 518kB ...
    TREFHTMX   (time, lat, lon) float32 473MB ...
    time_bnds  (time, nbnd) object 29kB ...
Attributes: (12/25)
    ne:                30
    fv_nphys:          2
    title:             EAM History file information
    source:        

/tmp/ipykernel_8855/1121706500.py:14: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print('dims:     ', dict(_ds.dims))


In [17]:
# SAI files are CDF-5 (PnetCDF) format — must be downloaded locally and opened
# with engine='netcdf4' (the C netCDF library). No regridding needed; files are
# already on a 180x360 lat/lon grid.
# Adjust the rename/squeeze lines below if the diagnostic shows non-standard dims.

for member, member_num in ALISTAIR_MEMBERS.items():
    print(f'Processing scenario=G6-1.5K-SAI, member={member}')

    tmax_s3 = sorted(fs.glob(f'{ALISTAIR_S3_BASE}daily_T_max/{member_num}/*.nc'))
    tmin_s3 = sorted(fs.glob(f'{ALISTAIR_S3_BASE}daily_T_min/{member_num}/*.nc'))
    print(f'  Found {len(tmax_s3)} TREFHTMX files, {len(tmin_s3)} TREFHTMN files')

    print('  Downloading TREFHTMX...')
    tmax_local = download_files(tmax_s3, TEMP_DIR)
    print('  Downloading TREFHTMN...')
    tmin_local = download_files(tmin_s3, TEMP_DIR)

    tasmax_ds = xr.open_mfdataset(
        tmax_local, combine='nested', concat_dim='time',
        engine='netcdf4', chunks=CHUNKS,
        coords='minimal', compat='override'
    )
    tasmin_ds = xr.open_mfdataset(
        tmin_local, combine='nested', concat_dim='time',
        engine='netcdf4', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    # If diagnostic shows dim 't' instead of 'time', or extra 'ht' dim, uncomment:
    # tasmax_ds = tasmax_ds.rename({'t': 'time'}).squeeze('ht', drop=True)
    # tasmin_ds = tasmin_ds.rename({'t': 'time'}).squeeze('ht', drop=True)

    tasmax_ds = tasmax_ds.drop_duplicates('time').sel(time=slice('2035', '2084'))
    tasmin_ds = tasmin_ds.drop_duplicates('time').sel(time=slice('2035', '2084'))

    tx = tasmax_ds['TREFHTMX'].assign_attrs({'units': 'K'})
    tn = tasmin_ds['TREFHTMN'].assign_attrs({'units': 'K'})

    out_dir = join(OUTPUT_ROOT, 'G6-1.5K-SAI', member)
    os.makedirs(out_dir, exist_ok=True)

    # --- Pass 1: tasmax-based indices ---
    su, id_, txx, txn = dask.compute(
        xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS'),
        xclim.indices.ice_days(tx, thresh='0 degC', freq='YS'),
        xclim.indices.tx_max(tx, freq='YS'),
        xclim.indices.tx_min(tx, freq='YS'),
    )
    su.to_netcdf(join(out_dir, 'SU.nc'));   del su
    id_.to_netcdf(join(out_dir, 'ID.nc'));  del id_
    txx.to_netcdf(join(out_dir, 'TXX.nc')); del txx
    txn.to_netcdf(join(out_dir, 'TXN.nc')); del txn
    gc.collect()

    # --- Pass 2: tasmin-based indices ---
    fd, tn_, tnx, tnn = dask.compute(
        xclim.indices.frost_days(tn, thresh='0 degC', freq='YS'),
        xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS'),
        xclim.indices.tn_max(tn, freq='YS'),
        xclim.indices.tn_min(tn, freq='YS'),
    )
    fd.to_netcdf(join(out_dir, 'FD.nc'));   del fd
    tn_.to_netcdf(join(out_dir, 'TN.nc'));  del tn_
    tnx.to_netcdf(join(out_dir, 'TNX.nc')); del tnx
    tnn.to_netcdf(join(out_dir, 'TNN.nc')); del tnn

    del tasmax_ds, tasmin_ds, tx, tn
    gc.collect()

    for p in tmax_local + tmin_local:
        if os.path.exists(p):
            os.remove(p)

    print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=G6-1.5K-SAI, member=r1
  Found 7 TREFHTMX files, 7 TREFHTMN files
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-SAI/r1
Processing scenario=G6-1.5K-SAI, member=r2
  Found 7 TREFHTMX files, 7 TREFHTMN files
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-SAI/r2
Processing scenario=G6-1.5K-SAI, member=r3
  Found 7 TREFHTMX files, 7 TREFHTMN files
  -> Temperature indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-SAI/r3


## Precipitation indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| PRECT | PRCPTOT | Total rainfall | Annual sum of precipitation | mm | Fixed Index |
| PRECT | SDII | Simple daily intensity | Mean precipitation falling on days when PR $\geq$ 1 mm | mm | Fixed Index |
| PRECT | Rx1day | Wettest day | Annual maximum precipitation in a single day | mm | Fixed Index |
| PRECT | Rx5day | Wettest pentad | Annual maximum precipitation falling on 5 consecutive days | mm | Fixed Index |
| PRECT | CDD | Consecutive dry days | Longest spell of consecutive days when PR $\leq$ 1 mm | days per year | Fixed index/spell |
| PRECT | CWD | Consecutive wet days | Longest spell of consecutive days when PR $\geq$ 1 mm | days per year | Fixed index/spell |
| PRECT | R10mm | Heavy precipitation days | Number of days when precipitation $\geq$ 10 mm | days per year | Fixed threshold |
| PRECT | R20mm | Very heavy precipitation days | Number of days when precipitation $\geq$ 20 mm | days per year | Fixed threshold |

### G6-1.5K-HiLLA and SSP245 (reflective-persistent-prod-large, native grid → ncremap)

Precipitation variable: `PRECT` (total precipitation rate, m/s) — converted to mm/day by `to_mm_day()`.

In [5]:
for scenario, members in E3SM_SCENARIOS.items():
    for member, member_dir in members.items():
        print(f'Processing scenario={scenario}, member={member} ({member_dir})')

        prect_s3 = sorted(fs.glob(
            f'{E3SM_S3_BASE}{member_dir}/day/PRECT/gn/*/PRECT_*.nc'
        ))
        print(f'  Found {len(prect_s3)} PRECT files')

        print('  Downloading and regridding PRECT...')
        prect_local = download_regrid_files(prect_s3, TEMP_DIR, MAP_LOCAL)

        ds = xr.open_mfdataset(
            prect_local, combine='nested', concat_dim='time',
            chunks=CHUNKS, coords='minimal', compat='override'
        )

        # Deduplicate before slicing (SSP245 files span 2015–2084 with possible
        # overlapping timestamps at decade boundaries)
        ds = ds.drop_duplicates('time')

        ds = ds.sel(time=slice('2035', '2084'))
        pr = to_mm_day(ds['PRECT'], varname='PRECT')

        out_dir = join(OUTPUT_ROOT, scenario, member)
        os.makedirs(out_dir, exist_ok=True)

        # --- Pass 1: simple per-timestep aggregations ---
        prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
            xclim.indices.prcptot(pr, freq='YS'),
            xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
            xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
            xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
            xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
            xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
        )
        prcptot.to_netcdf(join(out_dir, 'PRCPTOT.nc')); del prcptot
        sdii.to_netcdf(join(out_dir, 'SDII.nc'));       del sdii
        rx1d.to_netcdf(join(out_dir, 'RX1D.nc'));       del rx1d
        rx5d.to_netcdf(join(out_dir, 'RX5D.nc'));       del rx5d
        r10mm.to_netcdf(join(out_dir, 'R10MM.nc'));     del r10mm
        r20mm.to_netcdf(join(out_dir, 'R20MM.nc'));     del r20mm
        gc.collect()

        # --- Pass 2: consecutive-spell indices (memory-intensive, kept separate) ---
        cdd, cwd = dask.compute(
            xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
            xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
        )
        cdd.to_netcdf(join(out_dir, 'CDD.nc')); del cdd
        cwd.to_netcdf(join(out_dir, 'CWD.nc')); del cwd

        del ds, pr
        gc.collect()

        # Clean up regridded temp files to free disk space
        for p in prect_local:
            if os.path.exists(p):
                os.remove(p)

        print(f'  -> Precipitation indices saved to {out_dir}')

Processing scenario=G6-1.5K-HiLLA, member=r1 (v3.LR.ssp245.g6_hilla.sai.0101)
  Found 10 PRECT files
    Regridding  PRECT_203501_203912.nc...
    Regridding  PRECT_204001_204412.nc...
    Regridding  PRECT_204501_204912.nc...
    Regridding  PRECT_205001_205412.nc...
    Regridding  PRECT_205501_205912.nc...
    Regridding  PRECT_206001_206412.nc...
    Regridding  PRECT_206501_206912.nc...
    Regridding  PRECT_207001_207412.nc...
    Regridding  PRECT_207501_207912.nc...
    Regridding  PRECT_208001_208412.nc...


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-HiLLA/r1
Processing scenario=G6-1.5K-HiLLA, member=r2 (v3.LR.ssp245.g6_hilla.sai.0151)
  Found 10 PRECT files
    Regridding  PRECT_203501_203912.nc...
    Regridding  PRECT_204001_204412.nc...
    Regridding  PRECT_204501_204912.nc...
    Regridding  PRECT_205001_205412.nc...
    Regridding  PRECT_205501_205912.nc...
    Regridding  PRECT_206001_206412.nc...
    Regridding  PRECT_206501_206912.nc...
    Regridding  PRECT_207001_207412.nc...
    Regridding  PRECT_207501_207912.nc...
    Regridding  PRECT_208001_208412.nc...


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-HiLLA/r2
Processing scenario=G6-1.5K-HiLLA, member=r3 (v3.LR.ssp245.g6_hilla.sai.0201)
  Found 10 PRECT files
    Regridding  PRECT_203501_203912.nc...
    Regridding  PRECT_204001_204412.nc...
    Regridding  PRECT_204501_204912.nc...
    Regridding  PRECT_205001_205412.nc...
    Regridding  PRECT_205501_205912.nc...
    Regridding  PRECT_206001_206412.nc...
    Regridding  PRECT_206501_206912.nc...
    Regridding  PRECT_207001_207412.nc...
    Regridding  PRECT_207501_207912.nc...
    Regridding  PRECT_208001_208412.nc...


...

/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/SSP245/r2
Processing scenario=SSP245, member=r3 (v3.LR.ssp245_0201)
  Found 13 PRECT files
    Regridding  PRECT_201501_201912.nc...
    Regridding  PRECT_202001_202412.nc...
    Regridding  PRECT_202501_202912.nc...
    Regridding  PRECT_203001_203412.nc...
    Regridding  PRECT_203501_203912.nc...
    Regridding  PRECT_204001_204412.nc...
    Regridding  PRECT_204501_204912.nc...
    Regridding  PRECT_205001_205412.nc...
    Regridding  PRECT_205501_205912.nc...
    Regridding  PRECT_206001_206412.nc...
    Regridding  PRECT_206501_206912.nc...
    Regridding  PRECT_207001_207412.nc...
    Regridding  PRECT_207501_207912.nc...


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

  -> Precipitation indices saved to ./out_data/ETCCDI_indices_annual/E3SMv3/SSP245/r3


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

### G6-1.5K-SAI (alistairduffey bucket, pre-regridded 180×360)

Precipitation variable: `PRECT` in `daily_pr/{member}/`  
If the file structure differs from `PRECT_*.nc`, check with `fs.glob()` and adjust the glob pattern.

In [6]:
for member, member_num in ALISTAIR_MEMBERS.items():
    print(f'Processing scenario=G6-1.5K-SAI, member={member}')

    pr_s3 = sorted(fs.glob(f'{ALISTAIR_S3_BASE}daily_pr/{member_num}/*.nc'))
    print(f'  Found {len(pr_s3)} PRECT files')

    print('  Downloading PRECT...')
    pr_local = download_files(pr_s3, TEMP_DIR)

    ds = xr.open_mfdataset(
        pr_local, combine='nested', concat_dim='time',
        engine='netcdf4', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    # If diagnostic shows dim 't' instead of 'time', or extra 'ht' dim, uncomment:
    # ds = ds.rename({'t': 'time'}).squeeze('ht', drop=True)

    ds = ds.drop_duplicates('time').sel(time=slice('2035', '2084'))
    pr = to_mm_day(ds['PRECT'], varname='PRECT')

    out_dir = join(OUTPUT_ROOT, 'G6-1.5K-SAI', member)
    os.makedirs(out_dir, exist_ok=True)

    # --- Pass 1: simple per-timestep aggregations ---
    prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
        xclim.indices.prcptot(pr, freq='YS'),
        xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
        xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
        xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
        xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
        xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
    )
    prcptot.to_netcdf(join(out_dir, 'PRCPTOT.nc')); del prcptot
    sdii.to_netcdf(join(out_dir, 'SDII.nc'));       del sdii
    rx1d.to_netcdf(join(out_dir, 'RX1D.nc'));       del rx1d
    rx5d.to_netcdf(join(out_dir, 'RX5D.nc'));       del rx5d
    r10mm.to_netcdf(join(out_dir, 'R10MM.nc'));     del r10mm
    r20mm.to_netcdf(join(out_dir, 'R20MM.nc'));     del r20mm
    gc.collect()

    # --- Pass 2: consecutive-spell indices (memory-intensive, kept separate) ---
    cdd, cwd = dask.compute(
        xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
        xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
    )
    cdd.to_netcdf(join(out_dir, 'CDD.nc')); del cdd
    cwd.to_netcdf(join(out_dir, 'CWD.nc')); del cwd

    del ds, pr
    gc.collect()

    for p in pr_local:
        if os.path.exists(p):
            os.remove(p)

    print(f'  -> Precipitation indices saved to {out_dir}')

Processing scenario=G6-1.5K-SAI, member=r1
  Found 7 PRECT files


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

PermissionError: [Errno 13] Permission denied: '/home/jovyan/ETCCDI_G6_SAI_HiLLA/out_data/ETCCDI_indices_annual/E3SMv3/G6-1.5K-SAI/r1/RX5D.nc'